<a href="https://colab.research.google.com/github/aditisankara/lucidrt/blob/feat%2Fdata-pipeline/src/notebooks/lucidrt_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "transformers>=4.44" "datasets>=2.20"

In [4]:
from datasets import load_dataset

raw = load_dataset("lighteval/med_paragraph_simplification")

print("splits + sizes:", {k: len(v) for k, v in raw.items()})
first = next(iter(raw.values()))
print("columns:", first.column_names)
print("\none example:")
print(first[0])

README.md:   0%|          | 0.00/526 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.48MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  744kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3568 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/411 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/480 [00:00<?, ? examples/s]

splits + sizes: {'train': 3568, 'validation': 411, 'test': 480}
columns: ['query', 'answer']

one example:
{'query': 'Two trials met the inclusion criteria. One compared 2% ketanserin ointment in polyethylene glycol (PEG) with PEG alone, used twice a day by 40 participants with arterial leg ulcers, for eight weeks or until healing, whichever was sooner. One compared topical application of blood-derived concentrated growth factor (CGF) with standard dressing (polyurethane film or foam); both applied weekly for six weeks by 61 participants with non-healing ulcers (venous, diabetic arterial, neuropathic, traumatic, or vasculitic). Both trials were small, reported results inadequately, and were of low methodological quality. Short follow-up times (six and eight weeks) meant it would be difficult to capture sufficient healing events to allow us to make comparisons between treatments. One trial demonstrated accelerated wound healing in the ketanserin group compared with the control group. In

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("vocab size:", tok.vocab_size, "| pad:", tok.pad_token, "| eos:", tok.eos_token)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

vocab size: 151643 | pad: <|endoftext|> | eos: <|im_end|>


In [7]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)

ex = raw["train"][0]
prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
full_text = prompt + ex["answer"] + tok.eos_token   # what the model will learn from: instruction+input, then the answer

enc = tok(full_text)

print("num tokens:", len(enc["input_ids"]))
print("first 20 ids:", enc["input_ids"][:20])
print("round-trip decode:", tok.decode(enc["input_ids"][:20]))

num tokens: 878
first 20 ids: [50, 70206, 279, 2701, 6457, 1467, 1119, 14396, 4128, 264, 8720, 646, 3535, 382, 52554, 1467, 510, 11613, 19080, 2270]
round-trip decode: Simplify the following medical text into plain language a patient can understand.

Medical text:
Two trials met
